# 02_pipeline — thin orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin and copy-and-edit friendly: users read source data into DataFrames, register those DataFrames with FabricOps guardrails, transform to target DataFrames, then register target outputs for validation, writing, and evidence capture.

Flow:

1. Run `00_env_config`
2. Import required functions
3. Select data agreement and register notebook
4. Read source data into DataFrames
5. Register source DataFrames with schema, drift, and DQ guardrails
6. Profile each registered source DataFrame
7. Check each source DataFrame against schema guardrails using its configured preset
8. Check each source DataFrame against data drift guardrails using its configured preset
9. Check each source DataFrame against DQ guardrails using its configured preset
10. Enrich each source profile with DQ result columns and write to `METADATA_DATA_CATALOGUE`
11. Transform to target DataFrames
12. Register target outputs and add audit columns
13. Check each target DataFrame against schema guardrails using its configured preset
14. Check each target DataFrame against data drift guardrails using its configured preset
15. Check each target DataFrame against DQ guardrails using its configured preset
16. Enrich each target profile with DQ result columns and write to `METADATA_DATA_CATALOGUE`
17. Write target tables
18. Capture many-to-many lineage tied to the notebook registry
19. Write runtime summary evidence to `METADATA_PIPELINE_RUNS`

## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.

In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports existing FabricOps callables directly for reads, profiling, guardrails, and writes. Metadata evidence helpers are imported only where they hide catalogue, lineage, and runtime-summary plumbing.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    enforce_dq_rules,
    get_selected_agreement,
    monitor_data_changes,
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    stop_if_failed,
    validate_schema,
    widget_select_agreement,
    write_lakehouse_table,
    write_catalogue_evidence,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)


## 3. Select data agreement and register notebook

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`.

In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "sample_agreement_pipeline"

widget_select_agreement(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


## 4. Read source data

FabricOps starts after the DataFrame exists. Read your source data using normal Spark code or the same helper functions used in `99_explore`.

For additional sources, copy this section, change the DataFrame name, and add the DataFrame to `SOURCE_DATASETS`.

In [ ]:
USE_SAMPLE_DATA = True
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "CHANGE_ME_dataset"

if USE_SAMPLE_DATA:
    df_minimal_source = read_lakehouse_csv(
        CONFIG,
        ENV_NAME,
        "source",
        "Files/sample/minimal_source.csv",
        spark_session=spark,
        header=True,
    )
else:
    df_minimal_source = read_lakehouse_table(
        CONFIG,
        ENV_NAME,
        "source",
        "CHANGE_ME_source_table",
        spark_session=spark,
    )

# Other common options:
# df_minimal_source = read_warehouse_table(CONFIG, ENV_NAME, "warehouse", "dbo", "CHANGE_ME_source_table", spark_session=spark)
# df_minimal_source = read_lakehouse_parquet(CONFIG, ENV_NAME, "source", "Files/CHANGE_ME/source_file.parquet", spark_session=spark)
# df_minimal_source = read_lakehouse_excel(CONFIG, ENV_NAME, "source", "Files/CHANGE_ME/source_file.xlsx", spark_session=spark)
# df_minimal_source = spark.read.table("CHANGE_ME_source_table")

## 5. Register source DataFrames with FabricOps guardrails

Register each DataFrame with the schema, drift, and DQ guardrails FabricOps should apply.

For another source, copy the source read block, create another DataFrame, then add another entry to `SOURCE_DATASETS`.

In [ ]:
SOURCE_DATASETS = {
    "minimal_source": {
        "df": df_minimal_source,
        "dataset_name": DATASET_NAME,
        "schema_preset": "allow_new_columns",
        "drift_preset": "changing_data",
        "dq_preset": "approved_rules",
        "expected_schema": {
            "customer_id": "bigint",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
        },
        "distribution_columns": ["status", "amount", "country_code"],
    },
    # Add more sources by reading a DataFrame above, then registering it here.
    # "another_source": {
    #     "df": df_another_source,
    #     "dataset_name": DATASET_NAME,
    #     "schema_preset": "strict",
    #     "drift_preset": "monitor_changing_data",
    #     "dq_preset": "skip",
    #     "expected_schema": {"id": "bigint"},
    # },
}

source_evidence_definitions = {
    source_name: {**{key: value for key, value in source_config.items() if key != "df"}, "stage": "source"}
    for source_name, source_config in SOURCE_DATASETS.items()
}


## 6. Profile each registered source DataFrame

Profiles are reused for drift baselines and catalogue evidence.


In [ ]:
source_profiles = {}
for source_name, source_config in SOURCE_DATASETS.items():
    source_df = source_config["df"]
    source_profiles[source_name] = profile_dataframe(
        source_df,
        table_name=source_name,
        exclude_columns=source_config.get("exclude_columns"),
        include_distributions=True,
        distribution_columns=source_config.get("distribution_columns"),
    )

## 7. Check each source schema

Each source uses its own `schema_preset`. Blocking failures stop before transformation.


In [ ]:
source_schema_results = {}
for source_name, source_config in SOURCE_DATASETS.items():
    source_df = source_config["df"]
    source_schema_results[source_name] = validate_schema(
        source_df,
        source_config["expected_schema"],
        preset=source_config.get("schema_preset", "strict"),
    )
    stop_if_failed(source_schema_results[source_name])


## 8. Check each source for data drift

Each source uses its own `drift_preset` and its own catalogue baseline.


In [ ]:
source_drift_results = {}
for source_name, source_config in SOURCE_DATASETS.items():
    source_df = source_config["df"]
    source_drift_results[source_name] = monitor_data_changes(
        spark,
        source_df,
        "METADATA_DATA_CATALOGUE",
        source_config.get("dataset_name", DATASET_NAME),
        source_name,
        stage="source",
        preset=source_config.get("drift_preset", "changing_data"),
        exclude_run_id=RUN_ID,
        distribution_columns=source_config.get("distribution_columns"),
    )
    stop_if_failed(source_drift_results[source_name])

## 9. Check each source with DQ guardrails

Each source uses its own `dq_preset`. Warning severity writes full data; error severity stops before downstream writes. No row filtering in v1.


In [ ]:
source_dq_results = {}
for source_name, source_config in SOURCE_DATASETS.items():
    source_df = source_config["df"]
    if source_config.get("dq_preset", "approved_rules") == "skip":
        source_dq_results[source_name] = {"status": "skipped", "can_continue": True, "checks": [], "message": "DQ guardrail skipped by preset."}
    else:
        source_dq_results[source_name] = enforce_dq_rules(
            source_df,
            CONFIG,
            ENV_NAME,
            source_config.get("dataset_name", DATASET_NAME),
            source_name,
            spark_session=spark,
        )
    print(source_dq_results[source_name])
    stop_if_failed(source_dq_results[source_name])

## 10. Write source catalogue evidence

FabricOps enriches profile rows with agreement, notebook registry, guardrail, and DQ result columns before writing `METADATA_DATA_CATALOGUE`.


In [ ]:
source_catalogue_status = write_catalogue_evidence(
    source_profiles,
    source_evidence_definitions,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    schema_results=source_schema_results,
    drift_results=source_drift_results,
    dq_results=source_dq_results,
)


## 11. Transform to target DataFrame

This is the main DIY section. Keep business logic here and reusable governance plumbing in package helpers.

For another output, copy the target block, change the transformation and output name, then add another entry to `TARGET_DATASETS`.

In [ ]:
df_minimal_target = (
    df_minimal_source
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= F.lit(100), F.lit("high"))
        .when(F.col("amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)

# Add more transformations or joins here. For many sources, use the friendly
# aliases above or access SOURCE_DATASETS["source_alias"]["df"].

## 12. Register target outputs and add audit columns

Register each target DataFrame with write settings and the guardrails FabricOps should apply before publication.

For another output, copy the target block, change the transformation and output name, then add another entry to `TARGET_DATASETS`.

In [ ]:
TARGET_DATASETS = {
    "minimal_target": {
        "df": df_minimal_target,
        "dataset_name": DATASET_NAME,
        "target_name": "minimal_target",
        "target_layer": "unified",
        "write_mode": "overwrite",
        "schema_preset": "strict",
        "drift_preset": "changing_data",
        "dq_preset": "approved_rules",
        "expected_schema": {
            "customer_id": "bigint",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
            "amount_band": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
        "distribution_columns": ["status", "amount", "amount_band", "country_code"],
    },
    # Add more targets by creating a DataFrame above, then registering it here.
}

AUDIT_CREATED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
for target_name, target_config in TARGET_DATASETS.items():
    target_config["df"] = (
        target_config["df"]
        .withColumn("_fabricops_run_id", F.lit(RUN_ID))
        .withColumn("_fabricops_pipeline_name", F.lit(PIPELINE_NAME))
        .withColumn("_fabricops_created_at", F.lit(AUDIT_CREATED_AT))
    )

target_evidence_definitions = {
    target_name: {
        **{key: value for key, value in target_config.items() if key != "df"},
        "table_name": target_config.get("target_name", target_name),
        "layer": target_config.get("target_layer", "unified"),
        "kind": target_config.get("target_kind", "lakehouse"),
        "mode": target_config.get("write_mode", "overwrite"),
        "stage": "target",
    }
    for target_name, target_config in TARGET_DATASETS.items()
}

df_minimal_target = TARGET_DATASETS["minimal_target"]["df"]

## 13. Check each target schema

Target checks mirror source checks and run before publication.


In [ ]:
target_schema_results = {}
for target_name, target_config in TARGET_DATASETS.items():
    target_df = target_config["df"]
    target_schema_results[target_name] = validate_schema(
        target_df,
        target_config["expected_schema"],
        preset=target_config.get("schema_preset", "strict"),
    )
    stop_if_failed(target_schema_results[target_name])

## 14. Check each target for data drift

Target drift compares proposed target DataFrames before any target write occurs.


In [ ]:
target_drift_results = {}
for target_name, target_config in TARGET_DATASETS.items():
    target_df = target_config["df"]
    target_drift_results[target_name] = monitor_data_changes(
        spark,
        target_df,
        "METADATA_DATA_CATALOGUE",
        target_config.get("dataset_name", DATASET_NAME),
        target_config.get("target_name", target_name),
        stage="target",
        preset=target_config.get("drift_preset", "changing_data"),
        exclude_run_id=RUN_ID,
        distribution_columns=target_config.get("distribution_columns"),
    )
    stop_if_failed(target_drift_results[target_name])

## 15. Check each target with DQ guardrails

Approved active DQ rules are evaluated as aggregate guardrails before writing full target datasets.


In [ ]:
# Warning severity writes full data; error severity stops before write. No row filtering in v1.
target_dq_results = {}
for target_name, target_config in TARGET_DATASETS.items():
    target_df = target_config["df"]
    if target_config.get("dq_preset", "approved_rules") == "skip":
        target_dq_results[target_name] = {"status": "skipped", "can_continue": True, "checks": [], "message": "DQ guardrail skipped by preset."}
    else:
        target_dq_results[target_name] = enforce_dq_rules(
            target_df,
            CONFIG,
            ENV_NAME,
            target_config.get("dataset_name", DATASET_NAME),
            target_config.get("target_name", target_name),
            spark_session=spark,
        )
    print(target_dq_results[target_name])
    stop_if_failed(target_dq_results[target_name])
    if "dataframe" in target_dq_results[target_name]:
        target_config["df"] = target_dq_results[target_name]["dataframe"]

df_minimal_target = TARGET_DATASETS["minimal_target"]["df"]

## 16. Write target catalogue evidence

FabricOps writes target profile evidence with DQ result columns using the same shape as source evidence.


In [ ]:
target_profiles = {}
for target_name, target_config in TARGET_DATASETS.items():
    target_df = target_config["df"]
    target_profiles[target_name] = profile_dataframe(
        target_df,
        table_name=target_config.get("target_name", target_name),
        exclude_columns=target_config.get("exclude_columns"),
        include_distributions=True,
        distribution_columns=target_config.get("distribution_columns"),
    )

target_catalogue_status = write_catalogue_evidence(
    target_profiles,
    target_evidence_definitions,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    schema_results=target_schema_results,
    drift_results=target_drift_results,
    dq_results=target_dq_results,
)

## 17. Write target tables

Only registered target DataFrames and write settings are passed to the writer.

In [ ]:
target_write_status = {}
for target_name, target_config in TARGET_DATASETS.items():
    target_df = target_config["df"]
    target_kind = target_config.get("target_kind", "lakehouse")
    target_layer = target_config.get("target_layer", "unified")
    target_table = target_config.get("target_name", target_name)
    target_mode = target_config.get("write_mode", "overwrite")

    if target_kind == "lakehouse":
        write_lakehouse_table(
            target_df,
            CONFIG,
            ENV_NAME,
            target_layer,
            target_table,
            mode=target_mode,
            partition_by=target_config.get("partition_by"),
            repartition_by=target_config.get("repartition_by"),
            overwrite_schema=target_config.get("overwrite_schema", target_mode == "overwrite"),
        )
    elif target_kind == "warehouse":
        write_warehouse_table(
            target_df,
            CONFIG,
            ENV_NAME,
            target_layer,
            target_config.get("schema", "dbo"),
            target_table,
            mode=target_mode,
        )
    else:
        raise ValueError(f"Unsupported target kind for {target_name}: {target_kind}")
    target_write_status[target_name] = "written"

## 18. Capture many-to-many lineage

Define source-to-target relationships at the business level. FabricOps builds and writes metadata rows tied to the selected notebook registration.


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": ["minimal_source"],
        "targets": ["minimal_target"],
        "operation": "derive amount band and publish governed target",
        "description": "Sample source rows are transformed into the minimal target output.",
    },
]

lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=DATASET_NAME,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 19. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.


In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_drift_results=source_drift_results,
    target_drift_results=target_drift_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
